In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings #embedding model -> gemini
from langchain_chroma import Chroma
from langchain_groq import ChatGroq #Output genration model -> groq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain.tools import tool
from langchain.agents import create_agent

In [4]:
#1.Load Data
loader = PyPDFLoader('../data/Crop-price.pdf')
docs = loader.load()
print(len(docs))

#2. Chunking
splitter = RecursiveCharacterTextSplitter(chunk_size=50 , chunk_overlap=10)
splitted_data = splitter.split_documents(docs)
print(len(splitted_data))

#3. Embedding Vector
# embeddings = GoogleGenerativeAIEmbeddings(model='gemini-embedding-001')
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

#4. Chroma db call that model and convert them into vectors and store insidei it
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings,
    persist_directory="./vector_db"
)

13
465


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4482.86it/s]


In [ ]:
#query phase

#this tool fetch most relevent context
@tool
def retrieval_tool(query:str):
    """
    This tool can help you to retrive the relevent data of PDF Document, and these pdf document have details about
    Crop price prediction model build by Ayush Rajput
    """

    data = vector_store.similarity_search(query=query, k=4)
    print(len(data),data[1].page_content) #second chunk data prints

    context=""
    for doc in data:
        context += doc.page_content + "\n\n"

    return context

# Prompt -> context + question that we will give to llm
system_prompt = """
You are a helpful assistant. Use retrieval_tool for questions
requiring information from the PDF documents.
If you don't know the answer, say "I don't know."
"""


#above context passed to llm
llm = ChatGroq(model='openai/gpt-oss-120b')

# Create Agent
agent = create_agent( #called tool again and again until most relevent asnwers found
    model=llm,
    tools=[retrieval_tool],
    system_prompt=system_prompt
)

query = "Who is AYush Rajput and What he build which crop price model he build?"
response = agent.invoke(
    {"messages":[{"role":"user" , "content":query}]},
    )

result = response["messages"][-1].content
print(result)


4 The proposed Crop Price Prediction System is
4 2.  R. Kumar, S. Singh, “Crop Market Price
4 | linkedin.com/in/ayush-rajput-199574287
4 Analysis for Crop Price Prediction in
4 Ayush Rajput
4 Analysis for Crop Price Prediction in
4 Analysis for Crop Price Prediction in
4 This project focuses on developing a Crop Price
4 This model is trained to predict crop prices
4 The backend of the Crop Price Prediction System
4 | linkedin.com/in/ayush-rajput-199574287
4 | linkedin.com/in/ayush-rajput-199574287
4 The proposed Crop Price Prediction System is
4 | linkedin.com/in/ayush-rajput-199574287
4 trends. A crop price prediction system can
4 The proposed Crop Price Prediction System is
4 This project focuses on developing a Crop Price
**Ayush Rajput**

- A young data‑science/ML enthusiast (LinkedIn profile – `linkedin.com/in/ayush‑rajput‑199574287`) who also maintains a public GitHub repository (`github.com/ayushrajput545`).  
- He was a member of the **Smart India Hackathon (SIH) 2025 – Nationa